# Phase 6 feature-ablation report

This student-run notebook starts Phase 6 by verifying the frozen Phase 4/5 inputs and recording the separate return and risk research questions. It does not train a model in step 6.1. A hash, fold, eligibility, allowlist, baseline, or conclusion mismatch stops execution before a report is written.

In [2]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / "src" / "model_training").is_dir():
    root = root.parent
if not (root / "src" / "model_training").is_dir():
    raise FileNotFoundError("Run this notebook from inside the HalalifyAI repository.")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

%cd {root}


D:\Arham\FYP\HalalifyAI


In [3]:
from src.model_training.feature_ablation.ablation_contract import (
    DEFAULT_OUTPUT,
    load_or_write_ablation_contract_report,
)

report = load_or_write_ablation_contract_report(DEFAULT_OUTPUT)
print(f"Status: {report['status']}")
print(f"Development folds: {len(report['development_folds'])}")
print(f"Allowlisted features: {len(report['feature_allowlist'])}")
print(f"Report written to: {DEFAULT_OUTPUT}")


Status: verified
Development folds: 4
Allowlisted features: 14
Report written to: reports\model_evaluation\feature_ablation\v1\ablation_contract.json


## What the output means

A `verified` status means the current dataset, metadata, Phase 4 protocol, Phase 5 baseline evidence, four folds, eligibility rules, feature allowlist, and Phase 5 conclusions match the frozen identities. It does **not** report a new model result or approve a feature. Continue to step 6.2 only after reviewing `reports/model_evaluation/feature_ablation/v1/ablation_contract.json`.

## Step 6.2 — Candidate-feature inventory

This section classifies all 56 columns in the frozen modeling dataset. Each of the 14 allowlisted market features receives a formula, lookback, first usable date, missingness count, information-availability rule, and Phase 6 group. Every other column is explicitly excluded with a reason. No fundamental is approved before the separate SEC readiness gate.

In [4]:
from src.model_training.feature_ablation.feature_inventory import (
    DEFAULT_INVENTORY_CSV,
    DEFAULT_INVENTORY_JSON,
    write_feature_inventory_report,
)

inventory_report, inventory = write_feature_inventory_report()
print(f"Status: {inventory_report['status']}")
print(f"Dataset columns classified: {inventory_report['column_counts']['dataset_columns']}")
print(f"Approved market candidates: {inventory_report['column_counts']['approved_market_candidates']}")
print(f"Eligible fundamentals: {inventory_report['column_counts']['eligible_fundamentals']}")
print(f"Excluded columns: {inventory_report['column_counts']['excluded_columns']}")
print(f"JSON report: {DEFAULT_INVENTORY_JSON}")
print(f"CSV inventory: {DEFAULT_INVENTORY_CSV}")
inventory


Status: verified
Dataset columns classified: 56
Approved market candidates: 14
Eligible fundamentals: 0
Excluded columns: 42
JSON report: reports\model_evaluation\feature_ablation\v1\feature_inventory.json
CSV inventory: reports\model_evaluation\feature_ablation\v1\feature_inventory.csv


,column,status,group,formula,lookback,source_inputs,earliest_usable_date,latest_available_date,missing_rows,missing_percent,information_availability_rule,exclusion_reason
0,return_1d,approved_market_candidate,downside_trend,adjusted_close(t) / adjusted_close(t-1) - 1,1 return interval (2 adjusted closes),adjusted_close,2016-01-05,2025-12-31,50,0.040004,Calculated after session t closes using only c...,
1,momentum_5d,approved_market_candidate,momentum,adjusted_close(t) / adjusted_close(t-5) - 1,5 return intervals (6 adjusted closes),adjusted_close,2016-01-11,2025-12-31,250,0.200021,Calculated after session t closes using only c...,
2,momentum_20d,approved_market_candidate,momentum,adjusted_close(t) / adjusted_close(t-20) - 1,20 return intervals (21 adjusted closes),adjusted_close,2016-02-02,2025-12-31,1000,0.800083,Calculated after session t closes using only c...,
3,momentum_60d,approved_market_candidate,momentum,adjusted_close(t) / adjusted_close(t-60) - 1,60 return intervals (61 adjusted closes),adjusted_close,2016-03-31,2025-12-31,3000,2.400250,Calculated after session t closes using only c...,
4,volatility_20d,approved_market_candidate,volatility,sample_std(return_1d[t-19:t]) * sqrt(252),20 daily returns (21 adjusted closes),adjusted_close,2016-02-02,2025-12-31,1000,0.800083,Calculated after session t closes using only c...,
5,volatility_60d,approved_market_candidate,volatility,sample_std(return_1d[t-59:t]) * sqrt(252),60 daily returns (61 adjusted closes),adjusted_close,2016-03-31,2025-12-31,3000,2.400250,Calculated after session t closes using only c...,
6,downside_dev_20d,approved_market_candidate,downside_trend,"sqrt(mean(min(return_1d, 0)^2 over 20 returns)...",20 daily returns (21 adjusted closes),adjusted_close,2016-02-02,2025-12-31,1000,0.800083,Calculated after session t closes using only c...,
7,drawdown_from_peak_60d,approved_market_candidate,downside_trend,adjusted_close(t) / max(adjusted_close[t-59:t]...,60 adjusted closes including t,adjusted_close,2016-03-30,2025-12-31,2950,2.360245,Calculated after session t closes using only c...,
8,price_to_sma20,approved_market_candidate,downside_trend,adjusted_close(t) / mean(adjusted_close[t-19:t...,20 adjusted closes including t,adjusted_close,2016-02-01,2025-12-31,950,0.760079,Calculated after session t closes using only c...,
9,price_to_sma50,approved_market_candidate,downside_trend,adjusted_close(t) / mean(adjusted_close[t-49:t...,50 adjusted closes including t,adjusted_close,2016-03-15,2025-12-31,2450,1.960204,Calculated after session t closes using only c...,


## Step 6.2 review gate

Confirm that the five market groups contain all 14 allowlisted features, `eligible_fundamentals` is empty, and every remaining column has an exclusion reason. The reported earliest date is the first non-missing observation anywhere in the frozen dataset; it does not mean that every ticker is usable on that date. No feature performance has been measured yet.

## Step 6.3 — Validate the ablation runner

This cell checks the runner plan against the verified contract and inventory. It does not fit models. The plan contains each market group by itself, four cumulative combinations, and five leave-one-group-out references. For later steps, call one task, one estimator, and one experiment at a time so every result remains easy to inspect.

In [ ]:
from src.model_training.feature_ablation.ablation_runner import (
    DEFAULT_RUN_PLAN,
    write_runner_validation_report,
)

runner_report = write_runner_validation_report()
print(f"Status: {runner_report['status']}")
print(f"Predeclared experiments: {len(runner_report['planned_experiments'])}")
print(f"Tasks: {', '.join(runner_report['tasks'])}")
print(f"Estimators: {', '.join(runner_report['estimators'])}")
print(f"Model fitting performed: {runner_report['model_fitting_performed']}")
print(f"Validation report: {DEFAULT_RUN_PLAN}")
runner_report['planned_experiments']


## Step 6.3 review gate

Check that the validation report says `validated`, lists fourteen experiments, and records `model_fitting_performed` as `false`. The runner used for later feature comparisons preserves Phase 4 rows, tunes on inner validation only, records four outer-fold results, and refuses to overwrite run evidence. Proceed to Step 6.4 only after these checks and the focused tests pass.